In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math


In [ ]:



# ============================================================================
# 1. BASIC SELF-ATTENTION FROM SCRATCH
# ============================================================================

class SimpleSelfAttention(nn.Module):
    """
    Simplest possible self-attention implementation.
    Educational purpose - shows core concept without complications.
    """
    
    def __init__(self, embed_dim):
        """
        Args:
            embed_dim: Dimension of input embeddings
        """
        super(SimpleSelfAttention, self).__init__()
        
        # Three linear transformations for Q, K, V
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        
        self.embed_dim = embed_dim
    
    def forward(self, x):
        """
        Args:
            x: Input tensor [batch_size, seq_len, embed_dim]
        
        Returns:
            output: Attention output [batch_size, seq_len, embed_dim]
            attention_weights: [batch_size, seq_len, seq_len]
        """
        # Step 1: Create Q, K, V
        Q = self.query(x)  # [batch, seq_len, embed_dim]
        K = self.key(x)    # [batch, seq_len, embed_dim]
        V = self.value(x)  # [batch, seq_len, embed_dim]
        
        # Step 2: Compute attention scores (Q @ K^T)
        # [batch, seq_len, embed_dim] @ [batch, embed_dim, seq_len]
        # -> [batch, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1))
        
        # Step 3: Scale scores
        scores = scores / math.sqrt(self.embed_dim)
        
        # Step 4: Apply softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # Step 5: Apply attention to values
        # [batch, seq_len, seq_len] @ [batch, seq_len, embed_dim]
        # -> [batch, seq_len, embed_dim]
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights


# ============================================================================
# 2. SCALED DOT-PRODUCT ATTENTION
# ============================================================================

def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Compute Scaled Dot-Product Attention.
    
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    
    Args:
        query: [batch, num_heads, seq_len, d_k]
        key: [batch, num_heads, seq_len, d_k]
        value: [batch, num_heads, seq_len, d_v]
        mask: Optional mask [batch, 1, 1, seq_len] or [batch, 1, seq_len, seq_len]
        dropout: Optional dropout layer
    
    Returns:
        output: [batch, num_heads, seq_len, d_v]
        attention_weights: [batch, num_heads, seq_len, seq_len]
    """
    d_k = query.size(-1)
    
    # Compute attention scores
    # [batch, heads, seq_len, d_k] @ [batch, heads, d_k, seq_len]
    # -> [batch, heads, seq_len, seq_len]
    scores = torch.matmul(query, key.transpose(-2, -1))
    
    # Scale
    scores = scores / math.sqrt(d_k)
    
    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Apply dropout if provided
    if dropout is not None:
        attention_weights = dropout(attention_weights)
    
    # Apply attention to values
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights


# ============================================================================
# 3. MULTI-HEAD SELF-ATTENTION
# ============================================================================

class MultiHeadSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention mechanism.
    
    Splits the embedding into multiple heads, computes attention for each head
    independently, then concatenates and projects the results.
    """
    
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        """
        Args:
            embed_dim: Total dimension of the model
            num_heads: Number of attention heads
            dropout: Dropout probability
        """
        super(MultiHeadSelfAttention, self).__init__()
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Linear projections for Q, K, V
        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        # Scaling factor
        self.scale = math.sqrt(self.head_dim)
    
    def forward(self, x, mask=None, return_attention=False):
        """
        Args:
            x: Input tensor [batch_size, seq_len, embed_dim]
            mask: Optional attention mask
            return_attention: Whether to return attention weights
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
            attention_weights: [batch_size, num_heads, seq_len, seq_len] (if return_attention=True)
        """
        batch_size, seq_len, embed_dim = x.size()
        
        # Linear projections
        Q = self.q_linear(x)  # [batch, seq_len, embed_dim]
        K = self.k_linear(x)
        V = self.v_linear(x)
        
        # Reshape and transpose for multi-head attention
        # [batch, seq_len, embed_dim] -> [batch, seq_len, num_heads, head_dim]
        # -> [batch, num_heads, seq_len, head_dim]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attention_weights, V)
        
        # Concatenate heads
        # [batch, num_heads, seq_len, head_dim] -> [batch, seq_len, num_heads, head_dim]
        # -> [batch, seq_len, embed_dim]
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, embed_dim)
        
        # Final linear projection
        output = self.out_proj(attn_output)
        
        if return_attention:
            return output, attention_weights
        return output


# ============================================================================
# 4. MASKED SELF-ATTENTION (CAUSAL)
# ============================================================================

class CausalSelfAttention(nn.Module):
    """
    Masked (Causal) Self-Attention for autoregressive models.
    Each position can only attend to previous positions.
    """
    
    def __init__(self, embed_dim, num_heads, dropout=0.1, max_seq_len=512):
        """
        Args:
            embed_dim: Embedding dimension
            num_heads: Number of attention heads
            dropout: Dropout probability
            max_seq_len: Maximum sequence length for causal mask
        """
        super(CausalSelfAttention, self).__init__()
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Linear transformations
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        # Create causal mask
        # Upper triangular matrix of -inf
        causal_mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1)
        causal_mask = causal_mask.masked_fill(causal_mask == 1, float('-inf'))
        self.register_buffer('causal_mask', causal_mask)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
        """
        batch_size, seq_len, _ = x.size()
        
        # Project to Q, K, V
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Apply causal mask
        scores = scores + self.causal_mask[:seq_len, :seq_len]
        
        # Softmax and dropout
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)
        
        # Apply attention to values
        output = torch.matmul(attention, V)
        
        # Reshape and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        output = self.out_proj(output)
        
        return output


# ============================================================================
# 5. ATTENTION WITH RELATIVE POSITION BIAS
# ============================================================================

class RelativePositionSelfAttention(nn.Module):
    """
    Self-Attention with relative position encodings.
    Instead of absolute positions, encodes relative distances between positions.
    """
    
    def __init__(self, embed_dim, num_heads, max_relative_position=32):
        """
        Args:
            embed_dim: Embedding dimension
            num_heads: Number of attention heads
            max_relative_position: Maximum relative position to encode
        """
        super(RelativePositionSelfAttention, self).__init__()
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.max_relative_position = max_relative_position
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # Relative position embeddings
        self.relative_position_embeddings = nn.Embedding(
            2 * max_relative_position + 1,
            self.head_dim
        )
    
    def _get_relative_positions(self, seq_len):
        """Generate relative position matrix."""
        positions = torch.arange(seq_len)
        relative_positions = positions.unsqueeze(0) - positions.unsqueeze(1)
        
        # Clip to max relative position
        relative_positions = torch.clamp(
            relative_positions,
            -self.max_relative_position,
            self.max_relative_position
        )
        
        # Shift to start from 0
        relative_positions = relative_positions + self.max_relative_position
        
        return relative_positions
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
        """
        batch_size, seq_len, _ = x.size()
        
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Content-based attention
        content_scores = torch.matmul(Q, K.transpose(-2, -1))
        
        # Position-based attention
        relative_positions = self._get_relative_positions(seq_len).to(x.device)
        position_embeddings = self.relative_position_embeddings(relative_positions)
        
        # Add position bias
        # This is simplified; full implementation would be more complex
        position_scores = torch.einsum('bhqd,qkd->bhqk', Q, position_embeddings)
        
        # Combine
        scores = (content_scores + position_scores) / math.sqrt(self.head_dim)
        
        attention = F.softmax(scores, dim=-1)
        output = torch.matmul(attention, V)
        
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        output = self.out_proj(output)
        
        return output


# ============================================================================
# 6. VISUALIZATION FUNCTIONS
# ============================================================================

def visualize_attention(attention_weights, tokens=None, head_idx=0, figsize=(10, 8)):
    """
    Visualize attention weights as a heatmap.
    
    Args:
        attention_weights: Tensor [batch, num_heads, seq_len, seq_len] or [seq_len, seq_len]
        tokens: List of token strings (optional)
        head_idx: Which attention head to visualize
        figsize: Figure size
    """
    # Handle different input shapes
    if attention_weights.dim() == 4:
        attn = attention_weights[0, head_idx].detach().cpu().numpy()
    elif attention_weights.dim() == 3:
        attn = attention_weights[0].detach().cpu().numpy()
    else:
        attn = attention_weights.detach().cpu().numpy()
    
    seq_len = attn.shape[0]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot heatmap
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    
    # Add colorbar
    plt.colorbar(im, ax=ax)
    
    # Set ticks
    if tokens is not None:
        ax.set_xticks(range(seq_len))
        ax.set_yticks(range(seq_len))
        ax.set_xticklabels(tokens, rotation=45, ha='right')
        ax.set_yticklabels(tokens)
    
    # Labels
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    ax.set_title(f'Attention Weights (Head {head_idx})')
    
    # Add grid
    ax.set_xticks(np.arange(seq_len) - 0.5, minor=True)
    ax.set_yticks(np.arange(seq_len) - 0.5, minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    return fig


def visualize_multi_head_attention(attention_weights, tokens=None, figsize=(15, 10)):
    """
    Visualize all attention heads.
    
    Args:
        attention_weights: [batch, num_heads, seq_len, seq_len]
        tokens: List of token strings
        figsize: Figure size
    """
    if attention_weights.dim() == 4:
        attn = attention_weights[0].detach().cpu().numpy()
    else:
        attn = attention_weights.detach().cpu().numpy()
    
    num_heads = attn.shape[0]
    seq_len = attn.shape[1]
    
    # Calculate grid size
    cols = 4
    rows = (num_heads + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    
    for head_idx in range(num_heads):
        ax = axes[head_idx]
        im = ax.imshow(attn[head_idx], cmap='Blues', aspect='auto')
        
        if tokens is not None:
            ax.set_xticks(range(seq_len))
            ax.set_yticks(range(seq_len))
            ax.set_xticklabels(tokens, rotation=90, fontsize=8)
            ax.set_yticklabels(tokens, fontsize=8)
        
        ax.set_title(f'Head {head_idx}')
        plt.colorbar(im, ax=ax)
    
    # Hide extra subplots
    for idx in range(num_heads, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    return fig


# ============================================================================
# 7. EXAMPLE USAGE AND TESTS
# ============================================================================

def example_basic_attention():
    """Example: Basic self-attention."""
    print("="*80)
    print("EXAMPLE 1: Basic Self-Attention")
    print("="*80)
    
    # Parameters
    batch_size = 2
    seq_len = 5
    embed_dim = 8
    
    # Create random input
    x = torch.randn(batch_size, seq_len, embed_dim)
    
    # Create attention layer
    attention = SimpleSelfAttention(embed_dim)
    
    # Forward pass
    output, weights = attention(x)
    
    print(f"\nInput shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Attention weights shape: {weights.shape}")
    
    print(f"\nAttention weights for first sample:")
    print(weights[0].detach().numpy())
    print(f"\nEach row sums to 1: {weights[0].sum(dim=1)}")
    
    return output, weights


def example_multi_head():
    """Example: Multi-head self-attention."""
    print("\n" + "="*80)
    print("EXAMPLE 2: Multi-Head Self-Attention")
    print("="*80)
    
    batch_size = 1
    seq_len = 6
    embed_dim = 64
    num_heads = 8
    
    x = torch.randn(batch_size, seq_len, embed_dim)
    
    attention = MultiHeadSelfAttention(embed_dim, num_heads)
    output, weights = attention(x, return_attention=True)
    
    print(f"\nInput shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Attention weights shape: {weights.shape}")
    print(f"  - Batch: {weights.shape[0]}")
    print(f"  - Heads: {weights.shape[1]}")
    print(f"  - Query positions: {weights.shape[2]}")
    print(f"  - Key positions: {weights.shape[3]}")
    
    return output, weights


def example_masked_attention():
    """Example: Masked (causal) self-attention."""
    print("\n" + "="*80)
    print("EXAMPLE 3: Masked Self-Attention")
    print("="*80)
    
    batch_size = 1
    seq_len = 5
    embed_dim = 32
    num_heads = 4
    
    x = torch.randn(batch_size, seq_len, embed_dim)
    
    # Regular attention
    regular_attn = MultiHeadSelfAttention(embed_dim, num_heads)
    regular_output, regular_weights = regular_attn(x, return_attention=True)
    
    # Causal attention
    causal_attn = CausalSelfAttention(embed_dim, num_heads)
    causal_output = causal_attn(x)
    
    print(f"\nInput shape: {x.shape}")
    print(f"\nRegular attention - can see all positions:")
    print(regular_weights[0, 0].detach().numpy())
    
    # For causal, we'd need to modify to return weights
    print(f"\nCausal attention - can only see past:")
    print("(Lower triangular pattern expected)")
    
    return regular_output, causal_output


def example_sentence_attention():
    """Example: Self-attention on actual sentence."""
    print("\n" + "="*80)
    print("EXAMPLE 4: Sentence Self-Attention with Visualization")
    print("="*80)
    
    # Sentence
    tokens = ["The", "cat", "sat", "on", "the", "mat"]
    seq_len = len(tokens)
    embed_dim = 16
    num_heads = 4
    
    # Create simple embeddings (in practice, would use learned embeddings)
    # Here we use random for demonstration
    torch.manual_seed(42)
    embeddings = torch.randn(1, seq_len, embed_dim)
    
    # Apply multi-head attention
    attention = MultiHeadSelfAttention(embed_dim, num_heads)
    output, weights = attention(embeddings, return_attention=True)
    
    print(f"Sentence: {' '.join(tokens)}")
    print(f"\nAttention weights for head 0:")
    print(weights[0, 0].detach().numpy())
    
    # Visualize
    fig = visualize_attention(weights, tokens=tokens, head_idx=0)
    plt.savefig('/tmp/attention_visualization.png', dpi=150, bbox_inches='tight')
    print(f"\nVisualization saved to /tmp/attention_visualization.png")
    
    # Show what each token attends to
    print(f"\nAttention patterns (Head 0):")
    attn_matrix = weights[0, 0].detach().numpy()
    for i, token in enumerate(tokens):
        top_indices = attn_matrix[i].argsort()[-3:][::-1]
        top_tokens = [tokens[j] for j in top_indices]
        top_weights = [attn_matrix[i, j] for j in top_indices]
        print(f"  '{token}' attends most to: {list(zip(top_tokens, [f'{w:.3f}' for w in top_weights]))}")
    
    return output, weights


# ============================================================================
# 8. PERFORMANCE COMPARISON
# ============================================================================

def compare_attention_variants():
    """Compare different attention implementations."""
    print("\n" + "="*80)
    print("EXAMPLE 5: Performance Comparison")
    print("="*80)
    
    import time
    
    batch_size = 32
    seq_len = 128
    embed_dim = 256
    num_heads = 8
    
    x = torch.randn(batch_size, seq_len, embed_dim)
    
    # Test simple attention
    simple_attn = SimpleSelfAttention(embed_dim)
    start = time.time()
    for _ in range(100):
        _ = simple_attn(x)
    simple_time = time.time() - start
    
    # Test multi-head attention
    multi_attn = MultiHeadSelfAttention(embed_dim, num_heads)
    start = time.time()
    for _ in range(100):
        _ = multi_attn(x)
    multi_time = time.time() - start
    
    # Test causal attention
    causal_attn = CausalSelfAttention(embed_dim, num_heads, max_seq_len=seq_len)
    start = time.time()
    for _ in range(100):
        _ = causal_attn(x)
    causal_time = time.time() - start
    
    print(f"\nTiming (100 iterations):")
    print(f"  Simple Attention: {simple_time:.3f}s")
    print(f"  Multi-Head Attention: {multi_time:.3f}s")
    print(f"  Causal Attention: {causal_time:.3f}s")
    
    # Parameter count
    simple_params = sum(p.numel() for p in simple_attn.parameters())
    multi_params = sum(p.numel() for p in multi_attn.parameters())
    causal_params = sum(p.numel() for p in causal_attn.parameters())
    
    print(f"\nParameter count:")
    print(f"  Simple Attention: {simple_params:,}")
    print(f"  Multi-Head Attention: {multi_params:,}")
    print(f"  Causal Attention: {causal_params:,}")


# ============================================================================
# 9. PRACTICAL APPLICATION: TEXT CLASSIFICATION
# ============================================================================

class TextClassifierWithAttention(nn.Module):
    """
    Simple text classifier using self-attention.
    """
    
    def __init__(self, vocab_size, embed_dim, num_heads, num_classes):
        super(TextClassifierWithAttention, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = MultiHeadSelfAttention(embed_dim, num_heads)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len] (token indices)
        
        Returns:
            logits: [batch_size, num_classes]
        """
        # Embed
        x = self.embedding(x)  # [batch, seq_len, embed_dim]
        
        # Self-attention
        x, _ = self.attention(x, return_attention=True)
        
        # Pool (mean over sequence)
        x = x.mean(dim=1)  # [batch, embed_dim]
        
        # Classify
        logits = self.fc(x)  # [batch, num_classes]
        
        return logits


def example_text_classification():
    """Example: Text classification with attention."""
    print("\n" + "="*80)
    print("EXAMPLE 6: Text Classification with Self-Attention")
    print("="*80)
    
    vocab_size = 1000
    embed_dim = 64
    num_heads = 4
    num_classes = 2
    batch_size = 8
    seq_len = 20
    
    # Create model
    model = TextClassifierWithAttention(vocab_size, embed_dim, num_heads, num_classes)
    
    # Random input (token indices)
    x = torch.randint(0, vocab_size, (batch_size, seq_len))
    
    # Forward pass
    logits = model(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output logits shape: {logits.shape}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    return model, logits


# ============================================================================
# 10. MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("SELF-ATTENTION: COMPLETE IMPLEMENTATION AND EXAMPLES")
    print("="*80)
    
    # Run all examples
    example_basic_attention()
    example_multi_head()
    example_masked_attention()
    example_sentence_attention()
    compare_attention_variants()
    example_text_classification()
    
    print("\n" + "="*80)
    print("All examples completed successfully!")
    print("="*80)


# ============================================================================
# 11. ADDITIONAL UTILITIES
# ============================================================================

class AttentionAnalyzer:
    """
    Utility class for analyzing attention patterns.
    """
    
    @staticmethod
    def get_attention_statistics(attention_weights):
        """
        Compute statistics about attention patterns.
        
        Args:
            attention_weights: [batch, num_heads, seq_len, seq_len]
        
        Returns:
            dict with various statistics
        """
        attn = attention_weights.detach()
        
        stats = {
            'mean': attn.mean().item(),
            'std': attn.std().item(),
            'min': attn.min().item(),
            'max': attn.max().item(),
            'entropy': -(attn * torch.log(attn + 1e-9)).sum(dim=-1).mean().item(),
        }
        
        # Self-attention percentage (diagonal)
        batch, heads, seq_len, _ = attn.shape
        diagonal_mask = torch.eye(seq_len).bool()
        self_attn = attn[:, :, diagonal_mask].mean().item()
        stats['self_attention_ratio'] = self_attn
        
        return stats
    
    @staticmethod
    def find_most_attended_tokens(attention_weights, tokens, top_k=3):
        """
        Find which tokens receive most attention.
        
        Args:
            attention_weights: [batch, num_heads, seq_len, seq_len]
            tokens: List of token strings
            top_k: How many top tokens to return
        
        Returns:
            List of (token, attention_score) tuples
        """
        # Sum attention received by each token (across all queries)
        attn = attention_weights[0, 0].detach()  # First batch, first head
        attention_received = attn.sum(dim=0).numpy()
        
        # Get top-k
        top_indices = attention_received.argsort()[-top_k:][::-1]
        results = [(tokens[i], attention_received[i]) for i in top_indices]
        
        return results


def demo_attention_analysis():
    """Demonstrate attention analysis tools."""
    print("\n" + "="*80)
    print("EXAMPLE 7: Attention Analysis")
    print("="*80)
    
    tokens = ["The", "quick", "brown", "fox", "jumps"]
    seq_len = len(tokens)
    embed_dim = 32
    num_heads = 4
    
    # Create embeddings and get attention
    x = torch.randn(1, seq_len, embed_dim)
    attention = MultiHeadSelfAttention(embed_dim, num_heads)
    _, weights = attention(x, return_attention=True)
    
    # Analyze
    analyzer = AttentionAnalyzer()
    
    stats = analyzer.get_attention_statistics(weights)
    print("\nAttention Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value:.4f}")
    
    top_tokens = analyzer.find_most_attended_tokens(weights, tokens, top_k=3)
    print("\nMost Attended Tokens:")
    for token, score in top_tokens:
        print(f"  '{token}': {score:.4f}")


if __name__ == "__main__":
    demo_attention_analysis()

In [ ]:
# Create self-attention layer
attention = SimpleSelfAttention(embed_dim=128)

# Input: [batch_size, seq_len, embed_dim]
x = torch.randn(32, 20, 128)

# Forward pass
output, weights = attention(x)

print(output.shape)  # [32, 20, 128]
print(weights.shape)  # [32, 20, 20]

In [ ]:
# Create multi-head attention
attention = MultiHeadSelfAttention(embed_dim=256, num_heads=8)

x = torch.randn(16, 50, 256)
output, weights = attention(x, return_attention=True)

print(output.shape)  # [16, 50, 256]
print(weights.shape)  # [16, 8, 50, 50]

In [ ]:
# For autoregressive models (GPT-style)
attention = CausalSelfAttention(embed_dim=512, num_heads=8)

x = torch.randn(8, 100, 512)
output = attention(x)

# Each position can only see previous positions

In [ ]:
# Visualize attention patterns
tokens = ["I", "love", "deep", "learning"]
embeddings = torch.randn(1, 4, 64)

attention = MultiHeadSelfAttention(64, num_heads=4)
_, weights = attention(embeddings, return_attention=True)

# Visualize single head
visualize_attention(weights, tokens=tokens, head_idx=0)

# Visualize all heads
visualize_multi_head_attention(weights, tokens=tokens)

In [ ]:
# Masking
# For padding
scores = scores.masked_fill(mask == 0, -1e9)

# For causal (autoregressive)
causal_mask = torch.triu(torch.ones(n, n), diagonal=1)
scores = scores + causal_mask * -1e9

In [ ]:
#Multi-Head Reshaping
# Split heads
x = x.view(batch, seq, num_heads, head_dim).transpose(1, 2)

# Combine heads
x = x.transpose(1, 2).contiguous().view(batch, seq, embed_dim)

In [ ]:
#out of memory
# Solution: Use gradient checkpointing
from torch.utils.checkpoint import checkpoint

output = checkpoint(attention, x)

In [ ]:
# Numerical Instability
# Solution: Add epsilon in softmax
attention = F.softmax(scores, dim=-1) + 1e-9

In [ ]:
#Slow Training

# Solution: Use compiled model (PyTorch 2.0+)
model = torch.compile(model)